# Stage C - Transfer Learning with MobileNetV2
This notebook trains a compact transfer learning model for CIFAR-10.

In [1]:


import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/a213696/Group-5-ML-Project.git"
REPO_NAME = "Group-5-ML-Project"

CURRENT = Path.cwd()

if (CURRENT / "src").exists():
    PROJECT_ROOT = CURRENT

elif CURRENT.name == "notebooks" and (CURRENT.parent / "src").exists():
    PROJECT_ROOT = CURRENT.parent

else:
    BASE_DIR = Path("/content") if Path("/content").exists() else CURRENT
    PROJECT_ROOT = BASE_DIR / REPO_NAME

    if not PROJECT_ROOT.exists():
        print("Cloning repository from GitHub...")
        subprocess.run(["git", "clone", REPO_URL, str(PROJECT_ROOT)], check=True)
    else:
        print("Repository already exists. Pulling latest version...")
        subprocess.run(["git", "-C", str(PROJECT_ROOT), "pull"], check=False)

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

FIG_DIR = PROJECT_ROOT / "artifacts" / "figures"
TABLE_DIR = PROJECT_ROOT / "artifacts" / "tables"
MODEL_DIR = PROJECT_ROOT / "artifacts" / "models"
for d in [FIG_DIR, TABLE_DIR, MODEL_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Current working directory:", Path.cwd())
print("src exists:", (PROJECT_ROOT / "src").exists())
print("notebooks exists:", (PROJECT_ROOT / "notebooks").exists())

if not (PROJECT_ROOT / "src").exists():
    raise FileNotFoundError("The src folder is missing. Upload src to GitHub or clone the full project repository, not only one notebook.")

import numpy as np, pandas as pd, json
import tensorflow as tf
from src.data import get_image_arrays, scale_to_float, set_seeds
from src.metrics import compute_metrics, classification_report_df, confusion_matrix_array
from src import viz
set_seeds(42)



Cloning repository from GitHub...
Project root: /content/Group-5-ML-Project
Current working directory: /content/Group-5-ML-Project
src exists: True
notebooks exists: False


In [2]:
(x_train, y_train), (x_val, y_val), (x_test, y_test) = get_image_arrays()
x_train_f = scale_to_float(x_train); x_val_f = scale_to_float(x_val); x_test_f = scale_to_float(x_test)
IMG_SIZE = 96
BATCH = 128
preprocess = tf.keras.applications.mobilenet_v2.preprocess_input

def prep(x, y):
    x = tf.image.resize(x, (IMG_SIZE, IMG_SIZE))
    x = preprocess(x * 255.0)
    return x, y

ds_train = tf.data.Dataset.from_tensor_slices((x_train_f, y_train)).shuffle(10000, seed=42).batch(BATCH).map(prep).prefetch(tf.data.AUTOTUNE)
ds_val = tf.data.Dataset.from_tensor_slices((x_val_f, y_val)).batch(BATCH).map(prep).prefetch(tf.data.AUTOTUNE)
ds_test = tf.data.Dataset.from_tensor_slices((x_test_f, y_test)).batch(BATCH).map(prep).prefetch(tf.data.AUTOTUNE)
print("Transfer learning input size:", IMG_SIZE)

170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 57s 0us/step
Transfer learning input size: 96


In [3]:
base = tf.keras.applications.MobileNetV2(include_top=False, weights="imagenet", input_shape=(IMG_SIZE, IMG_SIZE, 3))
base.trainable = False
inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = base(inputs, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.3)(x)
outputs = tf.keras.layers.Dense(10, activation="softmax")(x)
tl_model = tf.keras.Model(inputs, outputs, name="MobileNetV2_CIFAR10")
tl_model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
tl_model.summary()

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "MobileNetV2_CIFAR10"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 96, 96, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_96             │ (None, 3, 3, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 10)             │        12,810 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,270,794 (8.66 MB)

 Trainable params: 12,810 (50.04 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [ ]:
hist1 = tl_model.fit(ds_train, validation_data=ds_val, epochs=8, verbose=1)

base.trainable = True
for layer in base.layers[:-20]: layer.trainable = False
tl_model.compile(optimizer=tf.keras.optimizers.Adam(1e-4), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
hist2 = tl_model.fit(ds_train, validation_data=ds_val, epochs=8, verbose=1)
y_pred = tl_model.predict(ds_test, verbose=0).argmax(axis=1)
metrics = compute_metrics(y_test, y_pred, model_name="MobileNetV2 Transfer Learning")
print(metrics)
report_df = classification_report_df(y_test, y_pred)
display(report_df)
report_df.to_csv(TABLE_DIR / "stageC_tl_classification_report.csv")
cm = confusion_matrix_array(y_test, y_pred)
viz.plot_confusion_matrix(cm, title="Stage C MobileNetV2 Confusion Matrix", save_name="stageC_tl_confusion_matrix.png")
tl_model.save(MODEL_DIR / "stageC_mobilenetv2.keras")

Epoch 1/8
352/352 ━━━━━━━━━━━━━━━━━━━━ 351s 982ms/step - accuracy: 0.7474 - loss: 0.7580 - val_accuracy: 0.8480 - val_loss: 0.4345
Epoch 2/8
344/352 ━━━━━━━━━━━━━━━━━━━━ 6s 872ms/step - accuracy: 0.8274 - loss: 0.5095

In [ ]:
result_record = {
    "model": "MobileNetV2 (Transfer Learning)",
    "stage": "C",
    "input_size": f"{IMG_SIZE}x{IMG_SIZE}",
    "phase1_lr": 0.001,
    "phase2_lr": 0.0001,
    "layers_unfrozen": 20,
    "test_accuracy": float(metrics["accuracy"]),
    "macro_f1": float(metrics["macro_f1"]),
    "total_params": int(tl_model.count_params()),
}
with open(TABLE_DIR / "stageC_tl_result.json", "w") as f: json.dump(result_record, f, indent=2)
print("Transfer learning result saved.")

## Transfer learning interpretation
MobileNetV2 starts with visual filters learned from a much larger image dataset, so it does not need to learn basic edges and textures from scratch. This usually improves performance on CIFAR-10, but it also adds dependency on a larger model and extra preprocessing such as resizing the image.